# 02 — Features: what each hospital learns from

Builds the modelling table shared by the forecaster (03), the sweep (04) and the evaluation (05).
Kept in its own notebook because every later stage reads exactly this file — one definition,
one artifact, no drift.

**Every feature is known before the month it predicts.** Targets are the log of monthly demand,
z-scored **per series using train months only** (asserted below — this is the leakage guard).

- lags of demand at t-1, t-2, t-3, t-12
- backlog and throughput at t-1 (the inventory position going in)
- month sin/cos (seasonality: December trough, March peak)

**Splits are dates, not counts** — train <= 2024-09 · val <= 2025-03 · test <= 2026-03 — so
results are directly comparable with the 35-trust `src/` run.

`lag12` consumes the first 12 months, so the first *modelled* month is **2022-04** — the month
national demand became stationary after COVID. The window is COVID-free by construction.

In [1]:
import numpy as np
import pandas as pd

long = pd.read_csv("../data/processed/rtt_long_national.csv")
key = ["provider_code", "specialty_code", "month"]
ser = ["provider_code", "specialty_code"]

cnt = long.pivot_table(index=key, columns="part", values="total_all", aggfunc="sum").reset_index()
wide = pd.DataFrame({"provider_code": cnt["provider_code"], "specialty_code": cnt["specialty_code"],
                     "month": cnt["month"], "demand": cnt["Part_3"], "backlog": cnt["Part_2"],
                     "throughput": cnt["Part_1A"].fillna(0) + cnt["Part_1B"].fillna(0)})
months = sorted(long["month"].unique())
ok = wide.dropna(subset=["demand"]).query("demand > 0").groupby(ser)["month"].nunique()
wide = wide.set_index(ser).loc[ok[ok == len(months)].index].reset_index()
wide[["backlog", "throughput"]] = wide[["backlog", "throughput"]].fillna(0)
wide = wide.sort_values(ser + ["month"]).reset_index(drop=True)

g = wide.groupby(ser, sort=False)
f = wide[key].copy()
f["demand"] = wide["demand"]
f["y"] = np.log1p(wide["demand"])
for lag in [1, 2, 3, 12]:
    f[f"lag{lag}"] = g["demand"].shift(lag).pipe(np.log1p)
f["backlog"] = g["backlog"].shift(1).pipe(np.log1p)
f["throughput"] = g["throughput"].shift(1).pipe(np.log1p)
mnum = f["month"].str[5:7].astype(int)
f["month_sin"] = np.sin(2 * np.pi * mnum / 12)
f["month_cos"] = np.cos(2 * np.pi * mnum / 12)
f = f.dropna().reset_index(drop=True)
f["split"] = np.where(f["month"] <= "2024-09", "train", np.where(f["month"] <= "2025-03", "val", "test"))

In [2]:
# per-series standardization, statistics from TRAIN months only
tr = f[f["split"] == "train"]
stats = tr.groupby(ser).agg(mu=("y", "mean"), sd=("y", "std"),
                            mu_b=("backlog", "mean"), sd_b=("backlog", "std"),
                            mu_t=("throughput", "mean"), sd_t=("throughput", "std")).reset_index()
stats[["sd", "sd_b", "sd_t"]] = stats[["sd", "sd_b", "sd_t"]].clip(lower=1e-6)
f = f.merge(stats, on=ser)
f["y"] = (f["y"] - f["mu"]) / f["sd"]
for c in ["lag1", "lag2", "lag3", "lag12"]:
    f[c] = (f[c] - f["mu"]) / f["sd"]
f["backlog"] = (f["backlog"] - f["mu_b"]) / f["sd_b"]
f["throughput"] = (f["throughput"] - f["mu_t"]) / f["sd_t"]
f = f.drop(columns=["mu_b", "sd_b", "mu_t", "sd_t"])
f.to_csv("../data/processed/features_national.csv", index=False)

# leakage + alignment guards - these must never be relaxed
one = f.groupby(ser).size().idxmax()
s = f[(f["provider_code"] == one[0]) & (f["specialty_code"] == one[1])].sort_values("month")
assert np.allclose(s["lag1"].values[1:], s["y"].values[:-1]), "lag misaligned"
chk = f[f["split"] == "train"].groupby(ser)["y"].agg(["mean", "std"])
assert np.allclose(chk["mean"], 0, atol=1e-6) and np.allclose(chk["std"], 1, atol=1e-3), "stats not train-only"
assert f.groupby(ser)["split"].nunique().eq(3).all(), "a series is missing from a split"

n = {s: (f["split"] == s).sum() for s in ["train", "val", "test"]}
print(f"features: {len(f):,} rows | series {f.groupby(ser).ngroups:,} | "
      f"clients {f['provider_code'].nunique()} | departments {f['specialty_code'].nunique()}")
print(f"train/val/test: {n['train']:,} / {n['val']:,} / {n['test']:,}")
print(f"months modelled: {f['month'].min()} .. {f['month'].max()}")
print(f"per-client train rows: mean {n['train'] / f['provider_code'].nunique():.0f} | "
      f"min {f[f['split'] == 'train'].groupby('provider_code').size().min()}")

features: 99,648 rows | series 2,076 | clients 138 | departments 23
train/val/test: 62,280 / 12,456 / 24,912
months modelled: 2022-04 .. 2026-03
per-client train rows: mean 451 | min 30
